In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]="Rag-basic"

### Fixed Chunkings (PDF and MD)

In [ ]:
# PDF
from typing import List

import pypdf
from langchain_core.documents import Document

def load_pdf(file_path: str)-> List[Document]:
    reader = pypdf.PdfReader(file_path)
    docs = []
    for i, page in enumerate(reader.pages):
        docs.append(
            Document(
                page_content=page.extract_text() or "",
                metadata={"source":file_path, "page":i},
            )
        )

    return docs

docs = load_pdf(r"/path")
len(docs)

In [ ]:
# Character Split

from langchain_text_splitters import CharacterTextSplitter

def chunk_chars(docs: List[Document])-> List:
    splitter = CharacterTextSplitter(
        separator='.',
        chunk_size=500,
        chunk_overlap=50,
        add_start_index=True
    )

    all_split = splitter.split_documents(docs)
    return all_split

all_char_splits=chunk_chars(docs)
all_char_splits[0].page_content
# len(all_char_splits)

In [ ]:
# Recursive Character Split

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_recursive_chars(docs: List[Document])-> List:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        add_start_index=True
    )

    all_split = splitter.split_documents(docs)
    return all_split

all_rc_splits=chunk_recursive_chars(docs)
all_rc_splits[0].page_content
# len(all_rc_splits)

In [ ]:
# Markdown Header Split

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import TextLoader

def chunk_md(file_path: str) -> List[Document]:

    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    loader = TextLoader(file_path)
    docs = loader.load()
    print(docs)
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

    all_split = md_splitter.split_text(docs[0].page_content)
    return all_split

all_md_split=chunk_md(r"/path")
all_md_split

In [ ]:
# Markdown Text split

from langchain_text_splitters import MarkdownTextSplitter
from langchain_community.document_loaders import TextLoader

def chunk_md(file_path: str) -> List[Document]:

    loader = TextLoader(file_path)
    docs = loader.load()

    md_splitter = MarkdownTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    all_split = md_splitter.split_documents(docs)
    return all_split

all_md_split=chunk_md(r"/path")
all_md_split

## Context-Aware Chunking

In [ ]:
# NLTK Split

from langchain_text_splitters import NLTKTextSplitter


def chunk_nltk(docs):
    splitter = NLTKTextSplitter()
    all_split = splitter.split_documents(docs)

    return all_split

docs = load_pdf((r"/path"))
all_nltk_split = chunk_nltk(docs)
all_nltk_split

In [ ]:
# URL Load
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = ["https://workat.tech/core-cs/tutorial/processes-and-threads-os-6iboki1s2y3t/"]
loader = UnstructuredURLLoader(urls=urls)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True
)

all_split = splitter.split_documents(docs)
all_split